# Analysis of drops falling in cups, filmed from above

## 1. Documentation and Imports


Created on Wed Jan 27 2021

Author: Valentin Laplaud

The purpose of this code is to analyze drop experiment filmed from above. 
It should be able to detect drop exiting the cups and to track them to obtain 
a velocity using the trackpy library (http://soft-matter.github.io/trackpy/v0.4.2/)

In [1]:

from __future__ import division, unicode_literals, print_function 
 # for compatibility with Python 2 and 3

# plotting stuff
import matplotlib as mpl
import matplotlib.pyplot as plt

# numbers handling
import numpy as np
import pandas as pd

import pims # image handling for trackpy
import trackpy as tp # tracking library

from skimage import io

import random
random.seed() # random number generator (no seed)

DebugPlots = False # Display intermediate plots along the way

## 2. Define analysis functions

### 2.1 Loading stack and locating objects

Identification of objects on each image of the stack

_________
INPUTS:

filename is the path to the stack

end is the number of slice to analyze, end = 0 will analyze the full stack
_______________________

Fixed parameters :

In tp.locate/batch :

'21' is the estimate size (in px) of particles to track.

'minmass' is the minimal mass (integrated pixel value) for a particule, it is really high because image are 16bits, it may be easier to convert in 8bits before

invert=True means image grayscale is inverted as trackpy looks for bright objects

In [ ]:
def LoadAndLocate(filename,end):
    
    stack = pims.open(filename) # get the tiff stack
    
    if end == 0: # analyse all slices
        end = len(stack)
    
    features = tp.batch(stack[:end], 21, minmass=100000, invert=True); # equivalent to tp.locate but on all frames specified
       
    if DebugPlots == True: # Debug plots
        for nimg in range(1, end, 10):
            testF = tp.locate(stack[nimg], 21, minmass=100000, invert=True) # analyze only this image
            plt.figure
            plt.title('Frame number ' + str(nimg))
            tp.annotate(testF,stack[nimg]) # plot features identification
    
    return(stack, features)

### 2.2 Link located particles into trajectories

Creation of trajectories. 

In tp.link() : 

MaxDist is the maximum displacement allowed between two images. the higher it is the longer it wil take for the program to run.

'memory = 2' is the maximum of image a particule can disapear and still be considered part of the same trajectory when it reapears

In [ ]:
def MakeTraj(features, MaxDist, FrameRate, Scale):
    
    MaxDistpx = MaxDist*Scale # maximum displacement in pixels
    
    tracks = tp.link(features, MaxDistpx, memory = 2 )
    
    tracksFilt = tp.filter_stubs(tracks, 5) # removing trajectories that last less than 5 images
    
    Velocities = pd.DataFrame(data=None,columns=['Particle','MeanVel','VelStd']) # initializing velocites dataframe
    
    # Computes velocity for each particle
    for item in set(tracksFilt.particle):
        sub = tracksFilt[tracksFilt.particle==item]
        dx = np.diff(sub.x) # x displacement in px
        dy = np.diff(sub.y) # y displacement in px
        
        # velocity data in m/s
        vel = [[item, np.mean(np.sqrt(np.square(dx)+np.square(dy)))/Scale*FrameRate/1000, np.std(np.sqrt(np.square(dx)+np.square(dy)))/Scale*FrameRate/1000]]
        
        # generating dataframe with particle number, mean velocity and std of velocity
        v = pd.DataFrame(data=vel,columns=['Particle','MeanVel','VelStd'])
        
        Velocities = Velocities.append(v) # adding to global velocities dataframe
        
    # filter trajectories of immobile particles (v<0.2 m/s)
    
    V2 = Velocities[(Velocities['MeanVel']>0.2)]
    
    tracksFilt = tracksFilt[tracksFilt['particle'].isin(V2['Particle'])] # select only partciles present in V2
    
    return(tracks, tracksFilt, V2)

def PlotTraj(tracks, title):
    
    img = io.imread(P + '\\' + d + '.tif')

    background = img.min(0)
    
    minfig = plt.figure()
    plt.imshow(background,cmap='gray')
    minfig.savefig(P + '\MIN_' + d + '.png',dpi=300)
    plt.close(minfig)
    
    f = plt.figure()
    plt.imshow(background,cmap='gray')
    plt.title(title)
    
    particlelist = np.unique(tracks['particle'].to_numpy())
    
    for p in particlelist:
        
        particletrack = tracks[tracks['particle']==p]
        xtraj = particletrack['x'].to_numpy()
        ytraj = particletrack['y'].to_numpy()
        
        plt.plot(xtraj,ytraj)
    return(f)


## 3. Run analysis

### 3.1 Analyze and save data in dictionary

In [ ]:

P = r'd:\Users\Valentin Laplaud\PostDoc\Data\SplashCups\Manips\21.01.22_ObservationFromAbove'

ExpList = ['C4_75cm_S08-4','C4_75cm_S08-5','C4_75cm_S08-6','C5_75cm_S08-2','C5_75cm_S08-4']

end = 0 # number of slice to analyze per stack (0 = all)

MaxDist = 2 # Maximum distance from one image to next (mm)

FrameRate = 5000 # fps

Scale = 40 # px/mm

DataDict = {}

for s in ExpList:
    
    print('Analyzing ' + s + '...' , end=" ")  

    tp.quiet()
    stack, features = LoadAndLocate(P + '\\' + s + '.tif',end)
       
    T, TF, V= MakeTraj(features, MaxDist, FrameRate, Scale)
    
    DataDict[s] = {'tracks': T, 'filteredTracks': TF, 'velocities': V}
    
    print('Done.')
    print('')




### 3.2 Plot results

In [ ]:
AllV = pd.DataFrame(data=None,columns=['MeanVel','VelStd']) # initializing global velocity dataframe

for d in DataDict:

    # ploting trajectories on stack projection

    TF = DataDict[d]['filteredTracks']

    f = PlotTraj(TF, d)  
    
    f.savefig(P + '\FigTraj_' + d + '.png',dpi=300)

    if DebugPlots == True: # Debug plots
        plt.figure()
        plt.hist(V.MeanVel,bins=80)
        plt.xlabel('Velocity (m/s)')
        plt.ylabel('Count')
        plt.xlim(right=0.5)
        
        

    V = DataDict[d]['velocities']
        
    AllV = AllV.append(V[['MeanVel', 'VelStd']]) # adding to global velocities dataframe
    

plt.figure()
AllV[['MeanVel', 'VelStd']].boxplot()
AllV[['MeanVel', 'VelStd']].hist()


# Test zone

In [ ]:

AllV[['MeanVel']].hist(bins=70)


In [ ]:

im = io.imread(P + '\\' + d + '.tif')

im.shape
im2 = im.min(0)

im2.shape

plt.imshow(im2)

In [ ]:
tracks = TF

particlelist = np.unique(tracks['particle'].to_numpy())

    
    print(xtraj)
            

In [ ]:
a = range(1,8)
print(a(5))

In [ ]:
plt.figure(1)

In [ ]:
stack?

In [ ]:
stack[1].frame_no


